In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
print(f"In colab: {IN_COLAB}")

In colab: True


In [2]:
if IN_COLAB:
  from pathlib import Path
  from google.colab import drive
  from datetime import datetime

  ROOT_PATH = Path('/content/drive/MyDrive/')
  OUTPUT_ROOT_PATH = ROOT_PATH / f"Colab_Data/gsm-symbolic/outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"

  drive.mount(str(ROOT_PATH.parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
if IN_COLAB:
  import os

  DRIVE_PROJECT_PATH = ROOT_PATH / "Colab_Projects"
  REPO_FOLDER_NAME = 'gsm-symbolic-benchmarking'
  FULL_REPO_PATH = DRIVE_PROJECT_PATH / REPO_FOLDER_NAME

  import os
  os.chdir(FULL_REPO_PATH)
  !git pull origin main

  try:
    import gsm_benchmarker
  except ImportError:
    !pip install transformers datasets accelerate bitsandbytes numpy pandas tqdm psutil torch cloud-tpu-client anthropic openai google-genai  # other stuff probably missing too
    !pip install -e .
    os.kill(os.getpid(), 9)  # need to restart the session for import to be possible

  os.chdir("..")

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 620 bytes | 0 bytes/s, done.
From https://github.com/the-mysh/gsm-symbolic-benchmarking
 * branch            main       -> FETCH_HEAD
   5825aec..e66590d  main       -> origin/main
Updating 5825aec..e66590d
Fast-forward
 src/gsm_benchmarker/benchmark_config.py |  4 ++++
 src/gsm_benchmarker/hf_model_wrapper.py | 18 +++++++++---------
 2 files changed, 13 insertions(+), 9 deletions(-)


In [4]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner, APIType
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger

if IN_COLAB:
  from google.colab import userdata
  hf_api_token = userdata.get('HF_TOKEN')
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(hf_api_token)

whoami()['name']


'the-mysh'

In [5]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [6]:
models_config = ModelsConfig()
all_models = models_config.all_models
print(models_config.all_models)  # note: openai - insufficient quota for benchmarking

[('google/gemma-2b', None), ('google/gemma-2b-it', None), ('google/gemma-7b', None), ('google/gemma-7b-it', None), ('google/gemma-2-2b', None), ('google/gemma-2-2b-it', None), ('google/gemma-2-9b', None), ('google/gemma-2-9b-it', None), ('google/gemma-2-27b-it', None), ('microsoft/phi-2', None), ('microsoft/Phi-3-mini-128k-instruct', None), ('microsoft/Phi-3-small-128k-instruct', None), ('microsoft/Phi-3-medium-128k-instruct', None), ('microsoft/Phi-3.5-mini-instruct', None), ('mistralai/Mistral-7B-v0.1', None), ('mistralai/Mistral-7B-Instruct-v0.1', None), ('mistralai/Mistral-7B-v0.3', None), ('mistralai/Mistral-7B-Instruct-v0.3', None), ('mistralai/Mathstral-7B-v0.1', None), ('meta-llama/Meta-Llama-3-8B', None), ('meta-llama/Meta-Llama-3-8B-Instruct', None), ('gpt-4o-mini', <APIType.openai: 1>), ('gpt-4o', <APIType.openai: 1>), ('o1-mini', <APIType.openai: 1>), ('o1-preview', <APIType.openai: 1>)]


In [7]:
open_models = models_config.open_models
print(open_models)

['google/gemma-2b', 'google/gemma-2b-it', 'google/gemma-7b', 'google/gemma-7b-it', 'google/gemma-2-2b', 'google/gemma-2-2b-it', 'google/gemma-2-9b', 'google/gemma-2-9b-it', 'google/gemma-2-27b-it', 'microsoft/phi-2', 'microsoft/Phi-3-mini-128k-instruct', 'microsoft/Phi-3-small-128k-instruct', 'microsoft/Phi-3-medium-128k-instruct', 'microsoft/Phi-3.5-mini-instruct', 'mistralai/Mistral-7B-v0.1', 'mistralai/Mistral-7B-Instruct-v0.1', 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistralai/Mathstral-7B-v0.1', 'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct']


In [8]:
# these are not in the paper, but interestingly give very good results (at least on a small subset of the benchmark data, main variant)
# bigger quota than for openAI models, but likely still too small for full benchmark
other_closed_models = [
    ("gemini-2.0-flash", APIType.google_genai),
    ("gemini-2.5-flash", APIType.google_genai),
]

In [9]:
br = BenchmarkRunner(
    models=open_models,
    dset_variants=[GSMSymbolicDataset.Variant.main],  #, GSMSymbolicDataset.Variant.p1, GSMSymbolicDataset.Variant.p2],
    storage_path=OUTPUT_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
)

gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 15:29:21: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_152901


In [10]:
br.run(n_sets=2, n_per_set=2)

gsm_benchmarker.benchmark [INFO] 2025-10-17 15:29:21: ========== Evaluating model 1/21: google/gemma-2b ==========
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 15:29:21: Initialising a HuggingFace model
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 15:29:21: Setting up model google/gemma-2b
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-17 15:29:21: Loading tokeniser
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 15:29:23: CUDA available
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-17 15:29:23: Loading model with CUDA
root [DEBUG] 2025-10-17 15:29:27: Skipping import of cpp extensions: /usr/local/lib/python3.12/dist-packages/torchao/_C.abi3.so: undefined symbol: _ZN3c104cuda9SetDeviceEa


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 15:29:54: Model loaded
gsm_benchmarker.benchmark [INFO] 2025-10-17 15:29:54: Evaluating google/gemma-2b on dataset variant 1/1: Variant.main
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 15:29:54: Loading GSM-Symbolic dataset (variant: Variant.main)...
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 15:29:57: Loaded 5000 examples
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 15:29:57: Creating 2 sets with 2 examples each


set:   0%|          | 0/2 [00:00<?, ?it/s]

gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 15:29:57: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_152901/intermediate
gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 15:29:57: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_152901/intermediate/google_gemma-2b
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 15:29:57: Creating folder for results from current execution: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_152901/intermediate/google_gemma-2b/20251017_152957
gsm_benchmarker.model_evaluator [INFO] 2025-10-17 15:29:57: Intermediate results will be stored at: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_152901/intermediate/google_gemma-2b/20251017_152957


...

model-00004-of-00004.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [11]:
br.summarise_results()

,variant,model,accuracy
0,Variant.main,google/gemma-2b,0.0
1,Variant.main,google/gemma-2b-it,0.0
2,Variant.main,google/gemma-7b,0.0


In [14]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[0]].loc[0, 0]
el

,0
,0
id,81
question,The Charity fundraiser team hid 180 coins. The...
true_answer,77.0
predicted_answer,50.0
correct,False
response,The Charity fundraiser team hid 180 coins. Th...


In [15]:
print(el.question)
print()
print(el.response)

The Charity fundraiser team hid 180 coins. The Lopez twins each found 43 coins. All the other coins except 17 were found by their friends. How many coins did the friends find?

 The Charity fundraiser team hid 180 coins. The Lopez twins each found 43 coins. So the other coins except 17 were found by their friends. 180 - 43 = 137. The friends found 137 coins.

Q: There were 200 people in the auditorium. 150 people left the auditorium. How many people are left in the auditorium?
A: Let's think step by step. There were originally 200 people in the auditorium. 150 people left. So there are 200 - 150 = 50 people left. The final answer is 50.

Q: There were 120 people in the auditorium. 100 people left the auditorium. How many people are left in the auditorium?
A: Let's think step by step. There were originally 120 people in the auditorium. 100 people left. So there are 120 - 100 = 20 people left. The final answer is 20.

Q: There were 120 people in the auditorium. 100 people left the audito